# Model 1: Shooting Algorithm with Newton's Method (setup + pseudocode)

**Mara Dukuly & Adi Suresh**

**Purpose:** Develop and validate a Newton shooting solution approach for HHEI Model 1.

This notebook sets up the numerical solution of Model 1 (conflict, skill formation, and average human capital). It follows `model_1_sequential.tex`; equation numbers below refer to that document.

**Date started:** 2026-10-19 (file creation date; no Git history yet).

**Status:** Work in progress.

**Code style:** Use descriptive `snake_case` names, group imports, and keep code cells focused.

**What is implemented (and runs):**
1. Parameters and model primitives: output, CES skill formation and its derivatives.
2. The **forward map** $(s_t, m_t) \mapsto (s_{t+1}, m_{t+1})$, which iterates every FOC one period forward.
3. The **shooting function** (simulate $t = 0, \dots, T$) and the **terminal residual** $F(m_0)$.
4. The **steady-state residual** (17 equations) and a finite-difference Jacobian helper.

**What is left as pseudocode (nothing is solved here):**
- Solving for the steady state.
- **Damped Newton shooting**, the main solver suggested by our advisor.
- Continuation (homotopy) and multiple shooting / stacked Newton, for robustness.
- Gradient descent, Levenberg–Marquardt, the direct gradient method, and where SGD fits.
- The conflict experiment and cost-of-war measures.

| | Cass–Koopmans notebook | Model 1 |
|---|---|---|
| Unknowns at $t=0$ | $c_0$ (1) | $m_0 = (\lambda_0, \eta_{1..4,0}, \nu_0)$ (6) |
| Terminal targets | $K_{T+1}$ (1) | $s_{T+1} = (K, \bar H, H_{1..4})$ (6) |
| Search | bisection | **Newton** (bisection has no 6-D version) |

In [ ]:
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares, root

## 1. Model recap (the form the solver uses)

Planner's problem, with log utility and equal weights across ages:

$$\max \sum_{t=0}^{T}\beta^t\sum_{a=0}^{A}\ln C_{a,t}$$

subject to

$$\sum_a C_{a,t}+\sum_a X_{a,t}+K_{t+1}=Y_t+(1-\delta)K_t,\qquad Y_t=ZK_t^\alpha(\bar H_tL_t)^{1-\alpha}$$
$$H_{a+1,t+1}=\Omega_{a,t}\big[\phi_{a,t}X_{a,t}^{\rho_t}+\theta_{a,t}H_{a,t}^{\rho_t}+\psi_{a,t}P_t^{\rho_t}\big]^{1/\rho_t},\qquad \bar H_{t+1}=\tfrac1A H_{a^*,t}+\tfrac{A-1}{A}\bar H_t$$

**Working assumptions** (doc §1, to confirm together):
- (A1) The resource constraint drops the extra $+K_t$.
- (A2) Utility is $\ln C$.
- (A3) Parental human capital is $P_t = \bar H_t$.
- The entry bin is $a^* = 4$ (ages 12–14), and $A = 15$.

**Key implications of the FOCs** (doc §3.1):
- $C_{a,t} = c_t = 1/\lambda_t$ for every age.
- Only ages $a = 0..3$ invest; $X_{a,t} = 0$ for $a \ge 4$.

**States and co-states:**
- States (6, predetermined): $s_t = (K_t, \bar H_t, H_{1,t}, H_{2,t}, H_{3,t}, H_{4,t})$
- Co-states (6, jump): $m_t = (\lambda_t, \eta_{1,t}, \eta_{2,t}, \eta_{3,t}, \eta_{4,t}, \nu_t)$

## 2. Parameters

Parameters that vary by age and time are stored as arrays of shape `(T_sim + 2, n_inv)`, where `n_inv = a* = 4` (only the investing ages need CES parameters). **All values below are placeholders, not a calibration.** A period is 3 years, so annual $\beta$ and $\delta$ are compounded.

In [ ]:
class Params(NamedTuple):
    β: float          # discount factor (per 3-year period)
    α: float          # capital share
    δ: float          # depreciation (per 3-year period)
    A: int            # terminal age bin (ages 45-47)  -> A = 15
    a_star: int       # entry bin whose H feeds H̄       -> 4 (ages 12-14)
    T_sim: int        # simulation horizon (>> 15 to approximate infinite horizon)
    Z: np.ndarray     # TFP path,                 shape (T_sim+2,)
    L: np.ndarray     # labor path,               shape (T_sim+2,)
    h_new: np.ndarray # newborn human capital,    shape (T_sim+2,)
    ρ: np.ndarray     # CES substitution param,   shape (T_sim+2,)
    Ω: np.ndarray     # skill productivity,       shape (T_sim+2, a_star)
    φ: np.ndarray     # weight on investment X,   shape (T_sim+2, a_star)
    θ: np.ndarray     # weight on own H,          shape (T_sim+2, a_star)
    # ψ = 1 - φ - θ is the weight on parental H (computed, not stored)


def create_params(β_annual=0.96, α=0.33, δ_annual=0.08,
                  A=15, a_star=4, T_sim=80,
                  Z=1.0, L=1.0, h_new=1.0,
                  ρ=0.5, Ω=2.5, φ=0.4, θ=0.4):
    # NOTE: with these placeholders, Ω=1 gives a corner steady state (X*≈0);
    # Ω=2.5 gives an interior one. Replace all values with the calibration.
    '''Build a Params instance with constant (placeholder) paths.
    To make something age- or time-varying, overwrite the array afterwards,
    e.g. p.Ω[t_war, :] *= 0.7 for a war period.'''
    n_t = T_sim + 2
    full_t  = lambda v: np.full(n_t, float(v))
    full_ta = lambda v: np.full((n_t, a_star), float(v))
    β = β_annual ** 3
    δ = 1 - (1 - δ_annual) ** 3
    p = Params(β, α, δ, A, a_star, T_sim,
               full_t(Z), full_t(L), full_t(h_new), full_t(ρ),
               full_ta(Ω), full_ta(φ), full_ta(θ))
    ψ = 1 - p.φ - p.θ
    assert np.all(ψ > 0), "need ψ = 1-φ-θ > 0 (newborn X is pinned down through the P channel)"
    assert np.all(p.ρ < 1), "need ρ < 1"
    return p

p = create_params()
print(f"β = {p.β:.4f} per period, δ = {p.δ:.4f} per period, T_sim = {p.T_sim} periods (= {3*p.T_sim} years)")

## 3. Primitives

CES aggregator and its derivatives (doc eq. 8):

$$g_X=\phi\,\Omega^{\rho}(g/X)^{1-\rho},\quad g_H=\theta\,\Omega^{\rho}(g/H)^{1-\rho},\quad g_P=\psi\,\Omega^{\rho}(g/P)^{1-\rho}$$

These derivative formulas remain valid in the Cobb–Douglas limit $\rho \to 0$. Only $g$ itself needs a special case there.

In [ ]:
def output(K, Hb, t, p):
    return p.Z[t] * K**p.α * (Hb * p.L[t])**(1 - p.α)

def gross_return(K, Hb, t, p):
    'R_t = α Y_t / K_t + 1 - δ'
    return p.α * output(K, Hb, t, p) / K + 1 - p.δ

def mph(K, Hb, t, p):
    'Marginal product of average human capital: (1-α) Y_t / H̄_t'
    return (1 - p.α) * output(K, Hb, t, p) / Hb

def ces(X, H, P, t, a, p):
    'Skill formation g_{a,t}(X, H, P).'
    ρ, Ω, φ, θ = p.ρ[t], p.Ω[t, a], p.φ[t, a], p.θ[t, a]
    ψ = 1 - φ - θ
    if abs(ρ) < 1e-10:                                   # Cobb-Douglas limit
        return Ω * X**φ * H**θ * P**ψ
    return Ω * (φ * X**ρ + θ * H**ρ + ψ * P**ρ)**(1 / ρ)

def ces_derivs(X, H, P, t, a, p):
    'Returns g, g_X, g_H, g_P.'
    ρ, Ω, φ, θ = p.ρ[t], p.Ω[t, a], p.φ[t, a], p.θ[t, a]
    ψ = 1 - φ - θ
    g = ces(X, H, P, t, a, p)
    k = Ω**ρ
    return (g,
            φ * k * (g / X)**(1 - ρ),
            θ * k * (g / H)**(1 - ρ),
            ψ * k * (g / P)**(1 - ρ))

### Closed-form investment rules

* **Ages $a = 1..3$**, given $\eta_{a,t}$ (human-capital Euler equation, doc eq. 18):
$$X_{a,t}=H_{a,t}\Big[\tfrac{\phi}{\theta}\cdot\tfrac{\eta_{a,t}}{\lambda_t}\Big]^{1/(1-\rho)}$$
* **Newborns, $a = 0$**, given $\mathcal S_t=\eta_{1,t+1}g_{P,0,t}$ (doc eq. 19):
$$X_{0,t}=P_t\big[\beta\phi\,\mathcal S_t/(\psi\lambda_t)\big]^{1/(1-\rho)}$$
* **Any $a$**, given next period's $\eta_{a+1,t+1}$ (doc eq. 20). Used in the steady state and in stacked Newton.

In [ ]:
def X_from_eta_now(H, lam, eta_a, t, a, p):
    'Eq. (18): investment for a = 1..a*-1 given this period’s shadow value η_{a,t}.'
    ρ, φ, θ = p.ρ[t], p.φ[t, a], p.θ[t, a]
    return H * ((φ / θ) * eta_a / lam)**(1 / (1 - ρ))

def X_newborn(P, lam, S, t, p):
    'Eq. (19): newborn investment from S_t = η_{1,t+1} g_{P,0,t}.'
    ρ, φ, θ = p.ρ[t], p.φ[t, 0], p.θ[t, 0]
    ψ = 1 - φ - θ
    return P * (p.β * φ * S / (ψ * lam))**(1 / (1 - ρ))

def X_from_eta_next(H, P, lam, eta_next, t, a, p):
    'Eq. (20): investment given next period’s shadow value η_{a+1,t+1}. Returns nan if not interior.'
    ρ, Ω, φ, θ = p.ρ[t], p.Ω[t, a], p.φ[t, a], p.θ[t, a]
    ψ = 1 - φ - θ
    r = (lam / (p.β * eta_next * φ * Ω**ρ))**(1 / (1 - ρ))      # r = g/X
    denom = r**ρ - φ * Ω**ρ
    if abs(ρ) < 1e-10 or denom <= 0:
        return np.nan            # Cobb-Douglas or corner: handle separately
    return (Ω**ρ * (θ * H**ρ + ψ * P**ρ) / denom)**(1 / ρ)

## 4. The forward map (doc §5)

Given $(s_t, m_t)$, compute $(s_{t+1}, m_{t+1})$ in 7 explicit steps:

1. Compute $Y_t$, $P_t = \bar H_t$, and $c_t = 1/\lambda_t$.
2. For $a = 1, 2, 3$: compute $X_{a,t}$ from eq. 18, then $H_{a+1,t+1} = g$ and $\eta_{a+1,t+1} = \lambda_t / (\beta g_X)$.
3. Set $\nu_{t+1} = A\,\eta_{4,t}/\beta$.
4. For newborns: solve the $\nu_t$ FOC for $\mathcal S_t$, then compute $X_{0,t}$, $H_{1,t+1}$, and $\eta_{1,t+1}$.
5. Update $\bar H_{t+1} = H_{4,t}/A + \tfrac{A-1}{A}\bar H_t$.
6. Update capital: $K_{t+1} = Y_t + (1-\delta)K_t - (A+1)c_t - \sum_a X_{a,t}$.
7. Apply the consumption Euler equation: $\lambda_{t+1} = \lambda_t / (\beta R_{t+1})$.

If a guess implies $\mathcal S_t \le 0$ or $K_{t+1} \le 0$, the path is infeasible. The function flags this instead of crashing, because Newton's line search needs to know.

In [ ]:
class Infeasible(Exception):
    pass

def forward_step(t, s, m, p):
    '''One period of the forward recursion.
    s = [K, H̄, H1, H2, H3, H4]          (states at t)
    m = [λ, η1, η2, η3, η4, ν]          (co-states at t)
    Returns s_next, m_next, flows (dict of period-t allocations).'''
    β, A, a_s = p.β, p.A, p.a_star
    K, Hb = s[0], s[1]
    H = np.concatenate(([p.h_new[t]], s[2:]))        # H[0..a*], H[0] = newborn endowment
    lam, eta, nu = m[0], np.concatenate(([np.nan], m[1:1 + a_s])), m[1 + a_s]  # eta[1..a*]
    if lam <= 0 or K <= 0 or Hb <= 0:
        raise Infeasible(f"t={t}: nonpositive λ, K or H̄")

    # Step 1
    P   = Hb
    Y   = output(K, Hb, t, p)
    c   = 1 / lam
    X        = np.zeros(a_s)                 # X[0..a*-1]
    H_next   = np.zeros(a_s + 1)             # H_next[1..a*]
    eta_next = np.zeros(a_s + 1)             # eta_next[1..a*]
    gP       = np.zeros(a_s)

    # Step 2: ages 1..a*-1
    for a in range(1, a_s):
        X[a] = X_from_eta_now(H[a], lam, eta[a], t, a, p)
        g, gX, gH, gP[a] = ces_derivs(X[a], H[a], P, t, a, p)
        H_next[a + 1]   = g
        eta_next[a + 1] = lam / (β * gX)

    # Step 3
    nu_next = A * eta[a_s] / β

    # Step 4: newborns through the ν_t FOC
    S = (nu - lam * mph(K, Hb, t, p) - β * (A - 1) / A * nu_next) / β \
        - np.sum(eta_next[2:] * gP[1:])
    if S <= 0:
        raise Infeasible(f"t={t}: S_t = {S:.3g} <= 0 (newborn investment undefined)")
    X[0] = X_newborn(P, lam, S, t, p)
    g, gX, gH, gP[0] = ces_derivs(X[0], H[0], P, t, 0, p)
    H_next[1], eta_next[1] = g, lam / (β * gX)

    # Step 5
    Hb_next = H[a_s] / A + (A - 1) / A * Hb

    # Step 6
    K_next = Y + (1 - p.δ) * K - (A + 1) * c - X.sum()
    if K_next <= 0:
        raise Infeasible(f"t={t}: K_(t+1) = {K_next:.3g} <= 0")

    # Step 7
    lam_next = lam / (β * gross_return(K_next, Hb_next, t + 1, p))

    s_next = np.concatenate(([K_next, Hb_next], H_next[1:]))
    m_next = np.concatenate(([lam_next], eta_next[1:], [nu_next]))
    flows  = dict(Y=Y, c=c, X=X.copy(), I=K_next - (1 - p.δ) * K, S=S)
    return s_next, m_next, flows

## 5. Shooting and the terminal residual

$$F(x) = \log s_{T+1}\big(m_0 = e^{x};\, s_0\big) - \log s^{\text{target}},\qquad x \in \mathbb R^6$$

We iterate on $x = \log m_0$ so that every co-state stays positive. The target is the **steady state** (doc §6: shoot to $s^*$ with a long horizon instead of imposing $\eta_{T+1} = 0$).

In [ ]:
def shoot(m0, s0, p, T=None):
    '''Simulate t = 0..T from (s0, m0). Returns a dict of paths and a feasibility flag.
    Analogue of `shooting(pp, c0, k0, T)` in the Cass-Koopmans notebook.'''
    T = p.T_sim if T is None else T
    n_s, n_m = len(s0), len(m0)
    S_path = np.full((T + 2, n_s), np.nan); S_path[0] = s0
    M_path = np.full((T + 2, n_m), np.nan); M_path[0] = m0
    Y, c, X, I = (np.full(T + 1, np.nan), np.full(T + 1, np.nan),
                  np.full((T + 1, p.a_star), np.nan), np.full(T + 1, np.nan))
    feasible, msg = True, ""
    for t in range(T + 1):
        try:
            S_path[t + 1], M_path[t + 1], f = forward_step(t, S_path[t], M_path[t], p)
        except (Infeasible, FloatingPointError, OverflowError) as e:
            feasible, msg = False, str(e)
            break
        Y[t], c[t], X[t], I[t] = f['Y'], f['c'], f['X'], f['I']
    return dict(s=S_path, m=M_path, Y=Y, c=c, X=X, I=I, feasible=feasible, msg=msg)


BIG = 1e3   # residual returned for infeasible guesses (Newton’s line search backs off)

def terminal_residual(x, s0, s_target, p, T=None):
    'F(x): log-deviation of s_{T+1} from the target. x = log(m0).'
    with np.errstate(all='raise'):
        try:
            out = shoot(np.exp(x), s0, p, T)
        except FloatingPointError:
            return np.full(len(s0), BIG)
    if not out['feasible'] or not np.all(np.isfinite(out['s'][-1])):
        return np.full(len(s0), BIG)
    return np.log(out['s'][-1]) - np.log(s_target)


def jacobian_fd(F, x, h=1e-6):
    'Forward-difference Jacobian of F at x (6 extra function calls here).'
    F0 = F(x)
    J = np.empty((len(F0), len(x)))
    for j in range(len(x)):
        e = np.zeros_like(x); e[j] = h
        J[:, j] = (F(x + e) - F0) / h
    return J

## 6. Steady state: residual set up, solve left as pseudocode (doc §4)

There are 17 unknowns: $(K, \bar H, H_{1..4}, X_{0..3}, c, \lambda, \eta_{1..4}, \nu)$. All are positive, so we work in logs. The steady state is (i) the no-war benchmark, (ii) the shooting target $s^*$, and (iii) the initial guess $m^*$ for Newton.

In [ ]:
def steady_state_residual(z, p, t=-1):
    '''17 steady-state equations; z = log of the 17 unknowns.
    Uses the period-t parameter values (default: last period = long run).'''
    a_s, A, β, δ = p.a_star, p.A, p.β, p.δ
    v = np.exp(z)
    K, Hb = v[0], v[1]
    H = np.concatenate(([p.h_new[t]], v[2:2 + a_s]))         # H[0..4]
    X = v[2 + a_s: 2 + 2 * a_s]                                # X[0..3]
    c, lam = v[2 + 2 * a_s], v[3 + 2 * a_s]
    eta = np.concatenate(([np.nan], v[4 + 2 * a_s: 4 + 3 * a_s]))   # eta[1..4]
    nu = v[4 + 3 * a_s]

    Y = output(K, Hb, t, p)
    res = [gross_return(K, Hb, t, p) - 1 / β,                 # capital Euler
           Hb - H[a_s],                                       # H̄* = H*_{a*}
           (A + 1) * c - (Y - δ * K - X.sum()),               # resources
           lam * c - 1]                                       # λ = 1/c
    gP_sum = 0.0
    for a in range(a_s):
        g, gX, gH, gP = ces_derivs(X[a], H[a], Hb, t, a, p)
        res.append(np.log(H[a + 1]) - np.log(g))              # skill chain
        res.append(np.log(lam) - np.log(β * eta[a + 1] * gX)) # investment FOC
        if a >= 1:
            res.append(np.log(eta[a]) - np.log(β * eta[a + 1] * gH))   # skill value
        gP_sum += eta[a + 1] * gP
    res.append(np.log(eta[a_s]) - np.log(β * nu / A))          # η_{a*} = βν/A
    res.append(nu * (1 - β * (A - 1) / A) - lam * mph(K, Hb, t, p) - β * gP_sum)
    return np.array(res)

print("number of steady-state equations:", len(steady_state_residual(np.zeros(17), p)))

In [ ]:
def solve_steady_state(p):
    '''PSEUDOCODE (not run).

    1. Closed form for capital intensity:
           k_eff* = K/(H̄ L) = [ (1/β - 1 + δ) / (α Z) ]^(1/(α-1))
    2. Initial guess z0 (logs):
           H_a  = h_new for all a,  H̄ = h_new,  K = k_eff* · H̄ · L,
           X_a  = small share of Y (e.g. 0.02·Y),
           c    = (Y - δK - ΣX)/(A+1),   λ = 1/c,
           η_a  = λ (rough),             ν = λ·MPH / (1 - β(A-1)/A)
    3. sol = root(steady_state_residual, z0, args=(p,), method='hybr')
       If it fails, try method='lm', or solve first with ρ ≈ 0 (Cobb-Douglas) and
       continue in ρ toward the target value.
    4. Unpack exp(sol.x) into
           s_star = [K, H̄, H1, H2, H3, H4]
           m_star = [λ, η1, η2, η3, η4, ν]
    5. CHECK (strongest test of the forward map):
           forward_step(t, s_star, m_star, p) must return (s_star, m_star)
           to machine precision. If not, the forward map and the FOCs disagree.
    '''
    raise NotImplementedError("pseudocode: see docstring")

## 7. Newton shooting: the main solver (doc §6.2)

**Why not bisection?** Bisection needs one unknown, one target, and monotonicity. We have 6 and 6.

**Newton's method:**

$$x^{(k+1)} = x^{(k)} - \omega_k\,J(x^{(k)})^{-1}F(x^{(k)})$$

- Each iteration costs 1 + 6 simulations (for the finite-difference Jacobian).
- Convergence is quadratic once close: the number of correct digits roughly doubles each iteration.
- $\omega_k \in (0, 1]$ is a backtracking step. Halve it until the trial path is feasible and $\lVert F \rVert$ falls.

> **Diagnostic (doc §6.3), at placeholder values.** The forward map has 6 stable and 6 unstable eigenvalues, so Blanchard–Kahn holds and the saddle path is unique. But the largest root is ≈ 4.4 per period. Started from the *exact* steady state, round-off alone makes the path infeasible by $t \approx 22$, and the shooting Jacobian has a condition number of ~1e25 at $T = 10$.
>
> **So:** use `newton_shoot` below only on short horizons ($T \lesssim 5$–8), as a prototype. For the real transition, apply the same Newton iteration to the **stacked / multiple-shooting** system in §7.2.

In [ ]:
def newton_shoot(s0, s_target, x_guess, p, T=None, tol=1e-8, max_iter=50, verbose=True):
    '''PSEUDOCODE (not run). Damped Newton on F(x) = terminal_residual(x, ...).

    F = lambda x: terminal_residual(x, s0, s_target, p, T)

    x = x_guess                               # e.g. log(m_star), or the previous solution
    for k in range(max_iter):
        Fx = F(x)
        if max|Fx| < tol:  return x  (converged)

        J = jacobian_fd(F, x)                 # 6x6
        if cond(J) is huge: warn -> horizon too long for single shooting
                                    (use continuation in T, or multiple shooting)
        Δ = np.linalg.solve(J, -Fx)           # Newton direction

        ω = 1.0                               # backtracking line search
        while ω > 1e-8:
            F_trial = F(x + ω Δ)
            if F_trial is feasible (≠ BIG) and ||F_trial|| < (1 - 1e-4 ω) ||Fx||:
                break
            ω = ω / 2
        x = x + ω Δ
        if verbose: print k, ||Fx||, ω

    raise "did not converge" -> fall back to levenberg_marquardt_shoot or multiple shooting
    '''
    raise NotImplementedError("pseudocode: see docstring")

### 7.1 Making Newton robust: continuation (homotopy)

Forward shooting is a saddle-path problem: an error $\varepsilon$ in $m_0$ grows roughly like $\mu^{T}\varepsilon$, where $\mu > 1$ is the unstable eigenvalue. A long horizon makes $F$ very steep. The fix is to **start where the answer is known** and move in small steps:

- At $s_0 = s^*$, the solution is exactly $m_0 = m^*$, so $F = 0$ immediately.
- Move $s_0$ toward the war-affected state one step at a time, warm-starting each solve from the last.

In [ ]:
def solve_transition(s0_war, s_star, m_star, p, n_steps=10, T_schedule=(5, 20, 40, 80)):
    '''PSEUDOCODE (not run). Continuation in initial conditions, then in horizon.

    x = log(m_star)
    for τ in linspace(0, 1, n_steps + 1)[1:]:
        s0_τ = exp( (1-τ)·log(s_star) + τ·log(s0_war) )      # geometric path
        x = newton_shoot(s0_τ, s_star, x, p, T=T_schedule[0])     # short T: single shooting OK
        if Newton fails: halve the τ step and retry

    for T in T_schedule[1:]:                                    # lengthen the horizon
        x = stacked_newton(s0_war, s_star, m_star, p, T=T)       # warm-start from previous
                                                                # (single shooting fails past T≈5-8)
    return paths unpacked from the stacked solution x
    '''
    raise NotImplementedError("pseudocode: see docstring")

### 7.2 Main solver for long horizons: multiple shooting / stacked Newton

Single shooting cannot reach the horizon we need (see the diagnostic above), so **make every period's $(s_t, m_t)$ an unknown** and require consistency:

$$\mathcal G(z) = \begin{bmatrix} s_1 - \Phi_s(s_0, m_0) \\ m_1 - \Phi_m(s_0, m_0) \\ \vdots \\ s_{T+1} - s^* \end{bmatrix} = 0,\qquad z = (m_0, s_1, m_1, \ldots, s_{T+1})$$

- The system has about $12T$ unknowns, but the Jacobian is **block-banded** (period $t$ links only to $t+1$), so sparse Newton is fast.
- Nothing is simulated over long stretches, so errors do not compound.
- This is what Dynare's `perfect_foresight_solver` does.
- It can also impose the literal finite-$T$ conditions ($\eta_{T+1} = 0$) by replacing the $X$ FOC with a Fischer–Burmeister complementarity: $\varphi(a,b) = a + b - \sqrt{a^2 + b^2} = 0$, with $a = X$ and $b = \lambda - \beta\eta g_X$.

In [ ]:
def stacked_newton(s0, s_star, m_star, p, T, segment=1):   # segment ≤ ~5 periods
    '''PSEUDOCODE (not run).

    unknowns z = [m_0, (s_k, m_k) for each segment start k = segment, 2·segment, ..., T]
    G(z):
        for each segment [k, k+segment):
            (s, m) = (s_k, m_k)   (s_0 is data)
            for t in segment: (s, m) = forward_step(t, s, m, p)
            append (s, m) - (s_next_segment, m_next_segment)        # continuity
        append log(s_{T+1}) - log(s_star)                             # terminal
    initial guess: every (s_k, m_k) = (s_star, m_star)
    solve: root(G, z0, method='hybr')  or  least_squares(G, z0, jac_sparsity=banded_pattern)
    '''
    raise NotImplementedError("pseudocode: see docstring")

In [ ]:
def blanchard_kahn_check(s_star, m_star, p, t=5, h=1e-7):
    '''PSEUDOCODE (not run). Linearize the forward map at the steady state.

    y* = log([s_star, m_star])                       (12-vector)
    Φ(y) = log(forward_step(t, exp(y[:6]), exp(y[6:]), p)) stacked
    J[:, j] = (Φ(y* + h e_j) - Φ(y* - h e_j)) / (2h)    (central differences)
    μ = eigvals(J)
    expect: exactly 6 with |μ| > 1  (= number of co-states)  -> unique saddle path
            max |μ| tells you how fast single-shooting errors grow (~ max|μ|^T)
    (at the placeholder values: |μ|max ≈ 4.4, slowest stable root ≈ 0.99)
    '''
    raise NotImplementedError("pseudocode: see docstring")

## 8. Gradient descent, SGD, and when they are relevant (doc §7)

**Gradient descent on the shooting residual.** Minimize $\tfrac12\lVert F(x)\rVert^2$ with $x \leftarrow x - \eta\,J^\top F$.
- It works in principle, but it ignores that this is a *square root-finding* problem.
- It converges linearly at a rate set by the condition number of $J$, which the saddle-path instability makes enormous.
- Newton's step $J^{-1}F$ undoes that conditioning.

**Levenberg–Marquardt** is the principled middle ground: it behaves like gradient descent when far from the solution and like Newton when close. Use it as the fallback.

**SGD** needs an objective that is an average over many sampled terms. Model 1 is deterministic, so there is nothing to sample, and SGD would just be gradient descent with noise added. SGD *does* become relevant later in the project, for:
1. Calibrating $(\Omega, \phi, \theta, \rho)$ to many country-cohort moments, using mini-batches over observations.
2. A stochastic war/peace Markov version, where expectations are computed by simulation.

**Direct method.** Gradient-based optimization (not stochastic) is also useful as a cross-check: maximize welfare directly over $\{c_t, X_{a,t}\}$ with autodiff and L-BFGS-B. This never uses the FOCs.

In [ ]:
def gd_shoot(s0, s_target, x_guess, p, lr=1e-3, max_iter=100_000, tol=1e-8):
    '''PSEUDOCODE (not run). Plain gradient descent on L(x) = ½||F(x)||².
    Included for comparison; expect it to be far slower than Newton.

    x = x_guess
    for k in range(max_iter):
        Fx = F(x);  if ||Fx|| < tol: return x
        J  = jacobian_fd(F, x)
        grad = J.T @ Fx                        # ∇ ½||F||²
        x = x - lr * grad                      # (Adam / momentum help a little, not enough)
    Compare: iterations-to-tolerance vs newton_shoot  ->  a good figure for the appendix.
    '''
    raise NotImplementedError("pseudocode: see docstring")


def levenberg_marquardt_shoot(s0, s_target, x_guess, p, T=None):
    '''PSEUDOCODE (not run). Robust fallback, essentially one line:

    sol = least_squares(lambda x: terminal_residual(x, s0, s_target, p, T),
                        x_guess, method='lm', xtol=1e-12, ftol=1e-12)
    return sol.x
    '''
    raise NotImplementedError("pseudocode: see docstring")


def direct_welfare_max(s0, p, T):
    '''PSEUDOCODE (not run). Independent cross-check that does not use multipliers.

    decision vector d = [log c_0..c_T, X_{a,t} for a=0..3, t=0..T]   (X ≥ 0 via bounds)
    objective(d):
        simulate K, H, H̄ forward with the constraints (eqs. 2-5),
        heavy penalty if K_{t+1} ≤ 0,
        return -( Σ_t β^t (A+1) ln c_t  +  terminal value V*(s_{T+1}) )
    solve with scipy.optimize.minimize(method='L-BFGS-B', jac=autodiff gradient)
        (JAX: jax.grad(objective); finite differences are too slow at ~5T variables)
    CHECK: the paths should match the Newton shooting solution.
    '''
    raise NotImplementedError("pseudocode: see docstring")


def calibrate_sgd(data_moments, p0, n_epochs, batch_size):
    '''PSEUDOCODE (not run). Where SGD actually fits: the estimation stage.

    Θ = (Ω, φ, θ, ρ) by age/period;  loss(Θ) = mean_i || model_moment_i(Θ) - data_i ||²
    for epoch in range(n_epochs):
        for batch in shuffle(country-cohort observations, batch_size):
            for each i in batch: solve the model at Θ (newton_shoot, warm start)
            grad = ∂loss_batch/∂Θ          (implicit-function theorem on F=0, or autodiff)
            Θ = Adam_step(Θ, grad)
    '''
    raise NotImplementedError("pseudocode: see docstring")

## 9. The conflict experiment (doc §8)

1. **Baseline (no war):** start at $s_0 = s^*$. The economy stays at the steady state.
2. **War for $D$ periods before $t = 0$:**
   - Physical capital is rebuilt, so $K_0 = K^*$.
   - Human capital is not rebuilt: lower $\Omega_{a,t}$ (or block $X$) during the war, then run the skill chain forward to get $H_{a,0} < H^*_a$ and $\bar H_0 < \bar H^*$.
3. **Solve the transition** from $s_0^{\text{war}}$ back to $s^*$.
4. **Cost of conflict:**
   - Discounted output gap.
   - Consumption-equivalent welfare loss.
   - Half-life of the $\bar H$ gap.

In [ ]:
def war_initial_state(s_star, m_star, p, D=4, Ω_war_factor=0.6):
    '''PSEUDOCODE (not run).
    p_war = copy of p with Ω[t, :] *= Ω_war_factor for the D war periods
    start from (s_star, m_star) and run the skill chain (eq. 4) and H̄ rule (eq. 5)
    for D periods, holding X at its steady-state level (or at 0 if schools close)
    K_0 = K*  (the rebuilding assumption)
    return s0_war = [K*, H̄_0, H_1,0, ..., H_4,0]
    '''
    raise NotImplementedError("pseudocode: see docstring")


def cost_of_conflict(path_war, s_star, Y_star, c_star, p):
    '''PSEUDOCODE (not run).
    output_gap   = Σ_t β^t (Y* - Y_t)
    welfare_loss = μ solving  Σ_t β^t (A+1) ln((1-μ) c*) = Σ_t β^t (A+1) ln c_t
                   -> with log utility: ln(1-μ) = Σ β^t ln(c_t/c*) / Σ β^t
    half_life    = first t with (H̄* - H̄_t) ≤ ½ (H̄* - H̄_0)
    '''
    raise NotImplementedError("pseudocode: see docstring")

## 10. Tests to run once the solvers are filled in

| Test | What it catches |
|---|---|
| `forward_step(s*, m*)` returns `(s*, m*)` | any mismatch between the forward map and the FOCs |
| Newton from `s0 = s*` returns `m*` in 0 iterations | residual or terminal-condition bugs |
| Set $\phi = 0$ (no investment) and fix the $\bar H$ path: matches the Cass–Koopmans bisection notebook | Euler and capital block |
| `jacobian_fd` vs. a complex-step or JAX Jacobian | finite-difference step size |
| Direct welfare maximization paths = shooting paths | sign and timing errors in the multipliers |
| Doubling `T_sim` leaves the first 16 periods unchanged | horizon is long enough |
| `blanchard_kahn_check`: 6 eigenvalues outside the unit circle | determinacy at the calibration |

### Smoke test

The cell below runs the forward map once and the residual once at an **arbitrary** guess. It only checks that the code executes; it is not a solution. An arbitrary guess is usually infeasible within a few periods, which is exactly why Newton needs the steady-state warm start and continuation.

In [ ]:
s0_test = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0])     # [K, H̄, H1..H4]  (arbitrary)
m0_test = np.array([1.0, 0.5, 0.5, 0.5, 0.5, 2.0])     # [λ, η1..η4, ν]  (arbitrary)

try:
    s1, m1, f0 = forward_step(0, s0_test, m0_test, p)
    print("one step OK:  s1 =", np.round(s1, 4))
    print("              m1 =", np.round(m1, 4))
except Infeasible as e:
    print("first step infeasible at this guess:", e)

out = shoot(m0_test, s0_test, p, T=10)
print("10-period shoot feasible:", out['feasible'], "|", out['msg'])
print("F(log m0) =", np.round(terminal_residual(np.log(m0_test), s0_test, s0_test, p, T=10), 3))